In [1]:
source ../modules/setup.tcl

lassign "2017 18" year day

aoc::get-puzzle $year $day 1
# aoc::get-puzzle $year $day 1 2
set input [string trim [aoc::get-input $year $day]]
jupyter::html "<h2>Input</h2>"
jupyter::display "text/plain" [string range $input 0 100]...;

(cached)


## --- Day 18: Duet ---




You discover a tablet containing some strange assembly code labeled simply "[Duet](https://en.wikipedia.org/wiki/Duet)". Rather than bother the sound card with it, you decide to run the code yourself. Unfortunately, you don't see any documentation, so you're left to figure out what the instructions mean on your own.




It seems like the assembly is meant to operate on a set of <b>registers</b> that are each named with a single letter and that can each hold a single [integer](https://en.wikipedia.org/wiki/Integer). You suppose each register should start with a value of `0`.




There aren't that many instructions, so it shouldn't be hard to figure out what they do.  Here's what you determine:




- 
`snd X` <b>plays a sound</b> with a frequency equal to the value of `X`.

- 
`set X Y` <b>sets</b> register `X` to the value of `Y`.

- 
`add X Y` <b>increases</b> register `X` by the value of `Y`.

- 
`mul X Y` sets register `X` to the result of <b>multiplying</b> the value contained in register `X` by the value of `Y`.

- 
`mod X Y` sets register `X` to the <b>remainder</b> of dividing the value contained in register `X` by the value of `Y` (that is, it sets `X` to the result of `X` [modulo](https://en.wikipedia.org/wiki/Modulo_operation) `Y`).

- 
`rcv X` <b>recovers</b> the frequency of the last sound played, but only when the value of `X` is not zero. (If it is zero, the command does nothing.)

- 
`jgz X Y` <b>jumps</b> with an offset of the value of `Y`, but only if the value of `X` is <b>greater than zero</b>. (An offset of `2` skips the next instruction, an offset of `-1` jumps to the previous instruction, and so on.)




Many of the instructions can take either a register (a single letter) or a number. The value of a register is the integer it contains; the value of a number is that number.




After each <b>jump</b> instruction, the program continues with the instruction to which the <b>jump</b> jumped. After any other instruction, the program continues with the next instruction. Continuing (or jumping) off either end of the program terminates it.




For example:



```
set a 1
add a 2
mul a a
mod a 5
snd a
set a 0
rcv a
jgz a -1
set a 1
jgz a -2

```



- The first four instructions set `a` to `1`, add `2` to it, square it, and then set it to itself modulo `5`, resulting in a value of `4`.

- Then, a sound with frequency `4` (the value of `a`) is played.

- After that, `a` is set to `0`, causing the subsequent `rcv` and `jgz` instructions to both be skipped (`rcv` because `a` is `0`, and `jgz` because `a` is not greater than `0`).

- Finally, `a` is set to `1`, causing the next `jgz` instruction to activate, jumping back two instructions to another jump, which jumps again to the `rcv`, which ultimately triggers the <b>recover</b> operation.




At the time the <b>recover</b> operation is executed, the frequency of the last sound played is `4`.




<b>What is the value of the recovered frequency</b> (the value of the most recently played sound) the <b>first</b> time a `rcv` instruction is executed with a non-zero value?




(cached)

Input

set i 31
set a 1
mul p 17
jgz p p
mul a 2
add i -1
jgz i -2
add a -1
set i 127
set p 316
mul p 8505
m...

In [2]:
proc _val {regsVar arg} {
    upvar $regsVar regs
    if {[string is integer $arg]} {
        return $arg
    } {
        if {[info exists regs($arg)]} {
            return $regs($arg)
        } else {
            return 0
        }
    }
}



proc aoc::part1 input {
    set pc 0
    interp alias {} val {} _val regs
    set prog [split [string trim $input] \n]
    while {1} {

       set cmd [lindex $prog $pc]
       set args [lassign $cmd op]
#        puts $cmd
       switch -exact $op {
           set {
               lassign $args reg arg
               set regs($reg) [val $arg]
           }
           mod {
               lassign $args reg arg
               set old [val $reg]
               set mod [val $arg]
               set regs($reg) [expr {$old % $mod}]
           }
            mul {
               lassign $args reg arg
               set old [val $reg]
               set factor [val $arg]
               set regs($reg) [* $old $factor]
           }
           add {
               lassign $args reg arg
               incr regs($reg) [val $arg]
               
           }
           snd {
               lassign $args arg
               lappend snds [val $arg]
           }
           rcv {
               lassign $args arg
#                puts $cmd:[val $arg]
               if {[val $arg] != 0} {
                   return [lindex $snds end]
               }
           }
           jgz {
               lassign $args arg offset
               if {[val $arg] > 0} {
                   incr pc $offset
                   incr pc -1
               }
           }
           default {
               return -code error "Unrecognized op $op"
           }

       }
       incr pc
#        parray regs
    }
}


# part1 {set a 1
# add a 2
# mul a a
# mod a 5
# snd a
# set a 0
# rcv a
# jgz a -1
# set a 1
# jgz a -2}



## Designed with coroutines

Each program runs a coroutine. In the coroutine the following steps occur:

* Initialize and yield
* Fill the received buffer with the argument of the first call
* When the recv buffer is empty, send all the collected values and yield.
* If the result of the yield has no additional values, we are in deadlock (this only works if only in program is sending data)


In [3]:
proc _run {input id} {
    yield
    set pc 0
    set rcvs [yield]
    set snds {}
    set regs(p) $id
    interp alias {} val {} _val regs
    set prog [split [string trim $input] \n]
    while {1} {

       set cmd [lindex $prog $pc]
       set args [lassign $cmd op]
#        puts $cmd
       switch -exact $op {
           set {
               lassign $args reg arg
               set regs($reg) [val $arg]
           }
           mod {
               lassign $args reg arg
               set old [val $reg]
               set mod [val $arg]
               set regs($reg) [expr {$old % $mod}]
           }
            mul {
               lassign $args reg arg
               set old [val $reg]
               set factor [val $arg]
               set regs($reg) [* $old $factor]
           }
           add {
               lassign $args reg arg
               incr regs($reg) [val $arg]
               
           }
           snd {
               lassign $args arg
               lappend snds [val $arg]
           }
           rcv {
               lassign $args reg
#                puts $cmd:[val $arg]
               if {[llength $rcvs] == 0} {
#                    puts "-> $snds"
#                    puts "send [llength $snds]" 
                   set rcvs [yield $snds]
                   set snds {}
#                    puts "received [llength $rcvs]"
                    
                   if {[llength $rcvs] == 0} {
                        return deadlock
                   } 
               }
#                puts "Read from buffer"
               set rcvs [lassign $rcvs regs($reg)]
           }
           jgz {
#                parray regs
               lassign $args arg offset
               if {[val $arg] > 0} {
                   incr pc [val $offset]
                   incr pc -1
               }
           }
           default {
               return -code error "Unrecognized op $op"
           }

       }
       incr pc
#        parray regs
    }
}

set test {snd 1
snd 2
snd p
rcv a
rcv b
rcv c
rcv d}

proc aoc::part2 input {
coroutine prog0 _run $input 0
coroutine prog1 _run $input 1

# coroutine prog0 _run $test 0
# coroutine prog1 _run $test 1

set cnt 0
set order {prog0 prog1}
set rcvs {}
while {1} {
    incr cnt
    if {$cnt == 10000000} break
    lassign $order run other
#     puts "Runnin $run"
    set rcvs [$run $rcvs]

    if {$rcvs eq "deadlock"} {
#         puts $run:$rcvs
        break
    }
    if {$run eq "prog1"} {
        incr snds [llength $rcvs]
    }
    set order [list $other $run]
    
}
return $snds
}

In [4]:
aoc::results

Part1	2951 (9317 us)
Part2	7366 (487891 us)


2951 7366